# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 282.13it/s]


2026-07-08 09:35:14.199 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:903 - Data batch-empirical estimation of propensity score.


2026-07-08 09:35:14.208 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:952 - Data prediction of expected reward based on gbm model.


2026-07-08 09:35:15.486 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1069 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-07-08 09:35:48.875 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-07-08 09:35:49.260 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-07-08 09:35:49.643 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.505987,0.474101,0.539637,0.016946,b-ipw,reward_0
1,0.505237,0.504402,0.506024,0.000410,dm,reward_0
2,0.509390,0.477309,0.541695,0.016441,dr,reward_0
3,0.505237,0.504411,0.506023,0.000411,dros-opt,reward_0
4,0.509390,0.476998,0.540970,0.016318,dros-pess,reward_0
5,0.509766,0.476435,0.543869,0.017353,ipw,reward_0
6,0.509245,0.475424,0.543334,0.017277,rep,reward_0
7,0.509386,0.477049,0.540757,0.016238,sndr,reward_0
8,0.509321,0.475170,0.543699,0.017401,snips,reward_0
9,0.509390,0.477029,0.541335,0.016567,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 310.76it/s]


2026-07-08 09:35:50.183 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1301 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<09:27,  1.76it/s]

...

2026-07-08 09:36:43.404 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-07-08 09:36:43.793 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-07-08 09:36:44.184 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.503579,0.464638,0.542040,0.019738,b-ipw,reward_0
1,0.505186,0.504434,0.505933,0.000384,dm,reward_0
2,0.512929,0.477615,0.549508,0.018428,dr,reward_0
3,0.505186,0.504413,0.505914,0.000382,dros-opt,reward_0
4,0.512929,0.477346,0.548495,0.018229,dros-pess,reward_0
5,0.514744,0.476028,0.557916,0.020545,ipw,reward_0
6,0.512826,0.474732,0.552902,0.020250,rep,reward_0
7,0.512901,0.477969,0.548068,0.018012,sndr,reward_0
8,0.512880,0.473518,0.553909,0.020487,snips,reward_0
9,0.512929,0.477225,0.548119,0.018117,sg-dr,reward_0
